# Bài 6: Máy vectơ hỗ trợ (SVM) - Học máy ứng dụng
**Giảng viên biên soạn:** ThS. Nguyễn Thái Anh  
**Trường Đại học Văn Lang - Khoa Công nghệ Thông tin**  
**Năm học:** 2026-2027  

---  
Tệp Google Colab này tổng hợp đầy đủ **Tất cả các Bước Thực hành (Mục 2 đến Mục 6)**, **Lời giải Bài tập (Mục 7)** và **Kết quả đầu ra trực quan**.

## 0. Chuẩn bị Môi trường và Dữ liệu Mẫu

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_digits, make_classification, make_moons
from sklearn.svm import SVC, LinearSVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
import time

os.makedirs('data', exist_ok=True)
os.makedirs('baitap06', exist_ok=True)

if not os.path.exists('data/sinh_vien.csv'):
    np.random.seed(17)
    gio_on = np.random.uniform(2, 30, 120)
    diem_gk = np.random.uniform(1, 10, 120)
    qua_mon = ((gio_on * 0.2 + diem_gk * 0.8 + np.random.normal(0, 1, 120)) > 5).astype(int)
    df_sv = pd.DataFrame({'gio_on': gio_on, 'diem_giua_ky': diem_gk, 'qua_mon': qua_mon})
    df_sv.to_csv('data/sinh_vien.csv', index=False)

if not os.path.exists('data/sinh_vien_mo_rong.csv'):
    df_sv = pd.read_csv('data/sinh_vien.csv')
    df_sv['chi_tieu'] = np.random.uniform(1000, 5000, len(df_sv))
    df_sv.to_csv('data/sinh_vien_mo_rong.csv', index=False)

if not os.path.exists('data/hai_vang_trang.csv'):
    X_m, y_m = make_moons(n_samples=300, noise=0.25, random_state=17)
    df_moon = pd.DataFrame({'x1': X_m[:, 0], 'x2': X_m[:, 1], 'nhan': y_m})
    df_moon.to_csv('data/hai_vang_trang.csv', index=False)

print('Đã chuẩn bị xong môi trường và dữ liệu!')

---  
# PHẦN I: CÁC BƯỚC THỰC HÀNH TRONG BÀI LAB

### Bước 1: Đường có lề rộng nhất (`g1_le_rong_nhat.py`)

In [ ]:
X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]], cluster_std=0.9, random_state=0)
y = 1 - y

svm = SVC(kernel="linear", C=1000)
svm.fit(X, y)

w = svm.coef_[0]
b = svm.intercept_[0]
print(f"w1 = {w[0]:.3f}, w2 = {w[1]:.3f}, b = {b:.3f}")
do_dai_w = np.sqrt(w[0]**2 + w[1]**2)
print(f"Độ rộng của lề = 2 / {do_dai_w:.4f} = {2/do_dai_w:.4f}\n")

print("Số vectơ hỗ trợ:", len(svm.support_))
for i in svm.support_:
    f = svm.decision_function(X[i:i+1])[0]
    print(f"  điểm {i:2d}: x1={X[i,0]:.2f}, x2={X[i,1]:.2f}, lớp {y[i]}, f(x)={f:+.3f}")
print()

f_tat_ca = svm.decision_function(X)
ngoai = np.setdiff1d(np.arange(len(X)), svm.support_)
nho_nhat = np.abs(f_tat_ca[ngoai]).min()
print(f"|f(x)| nhỏ nhất của các điểm khác: {nho_nhat:.3f}\n")

chi_sv = SVC(kernel="linear", C=1000)
chi_sv.fit(X[svm.support_], y[svm.support_])
w2 = chi_sv.coef_[0]
b2 = chi_sv.intercept_[0]
print("Học lại chỉ với vectơ hỗ trợ:")
print(f"w1 = {w2[0]:.3f}, w2 = {w2[1]:.3f}, b = {b2:.3f}\n")

moi = np.array([[3.0, 3.0], [5.0, 4.5]])
for diem, f, lop in zip(moi, svm.decision_function(moi), svm.predict(moi)):
    print(f"Điểm ({diem[0]}, {diem[1]}): f(x)={f:+.3f}, đoán lớp {lop}")

### Bước 2: Lề mềm, Tham số C và Chuẩn hóa (`g2_tham_so_c.py`)

In [ ]:
df = pd.read_csv("data/sinh_vien.csv")
X_sv = df[["gio_on", "diem_giua_ky"]]
y_sv = df["qua_mon"]
X_tr, X_te, y_tr, y_te = train_test_split(X_sv, y_sv, test_size=0.25, random_state=17, stratify=y_sv)

print(f"Tập học có {len(X_tr)} bạn")
print("     C  số vecto HT  độ rộng lề  tập học  tập kiểm tra")
for C in [0.01, 0.1, 1, 10, 100]:
    mh = make_pipeline(StandardScaler(), SVC(kernel="linear", C=C))
    mh.fit(X_tr, y_tr)
    svm_m = mh[-1]
    le = 2 / np.linalg.norm(svm_m.coef_[0])
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"{C:6.2f} {len(svm_m.support_):12d} {le:11.2f} {a_hoc:8.4f} {a_kt:13.4f}")
print()

df3 = pd.read_csv("data/sinh_vien_mo_rong.csv")
X3 = df3[["gio_on", "diem_giua_ky", "chi_tieu"]]
y3 = df3["qua_mon"]
X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y3, test_size=0.25, random_state=17, stratify=y3)

tho = SVC().fit(X3_tr, y3_tr)
chuan = make_pipeline(StandardScaler(), SVC()).fit(X3_tr, y3_tr)
print("Ba cột, kernel RBF mặc định")
print(f" Đoán bừa cả lớp qua : {y3_te.mean():.4f}")
print(f" Không chuẩn hóa     : {accuracy_score(y3_te, tho.predict(X3_te)):.4f}")
print(f" Có chuẩn hóa        : {accuracy_score(y3_te, chuan.predict(X3_te)):.4f}")

### Bước 3: Nâng chiều bằng tay và Kernel (`g3_kernel.py`)

In [ ]:
x_1d = np.array([-4.0, -3.4, -2.8, -1.2, -0.7, -0.2, 0.3, 0.8, 1.3, 2.2, 2.3, 2.9, 3.5, 4.0])
y_1d = (np.abs(x_1d) < 1.8).astype(int)
tot = max(((x_1d <= t).astype(int) == y_1d).mean() for t in x_1d)
print(f"Một ngưỡng trên trục x, tốt nhất đúng: {tot:.4f}")

X_moi = np.c_[x_1d, x_1d ** 2]
svm_1d = SVC(kernel="linear", C=1000).fit(X_moi, y_1d)
print(f"Thêm cột x^2, SVM tuyến tính đúng: {svm_1d.score(X_moi, y_1d):.4f}\n")

df_vt = pd.read_csv("data/hai_vang_trang.csv")
X_vt = df_vt[["x1", "x2"]]
y_vt = df_vt["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(X_vt, y_vt, test_size=0.25, random_state=17, stratify=y_vt)

mo_hinh = {
    "SVM kernel tuyến tính": SVC(kernel="linear"),
    "SVM kernel đa thức bậc 3": SVC(kernel="poly", degree=3),
    "SVM kernel RBF": SVC(kernel="rbf"),
    "Hồi quy logistic": LogisticRegression(),
    "KNN, k = 13": KNeighborsClassifier(n_neighbors=13),
}
print("Hai vầng trăng, accuracy trên tập kiểm tra")
for ten, mh in mo_hinh.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f"  {ten:26s} {acc:.4f}")
print()

rbf = mo_hinh["SVM kernel RBF"]
print(f"Số vectơ hỗ trợ của kernel RBF: {len(rbf.support_)} trên {len(X_tr)} điểm học")

### Bước 4: Chọn C và Gamma bằng GridSearchCV (`g4_chon_tham_so.py`)

In [ ]:
df_vt = pd.read_csv("data/hai_vang_trang.csv")
X_vt = df_vt[["x1", "x2"]]
y_vt = df_vt["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(X_vt, y_vt, test_size=0.25, random_state=17, stratify=y_vt)

print("gamma   tập học  tập kiểm tra  số vecto HT")
for g in [0.1, 1, 10, 100]:
    mh = SVC(kernel="rbf", gamma=g).fit(X_tr, y_tr)
    a_hoc = accuracy_score(y_tr, mh.predict(X_tr))
    a_kt = accuracy_score(y_te, mh.predict(X_te))
    print(f"{g:5.1f}   {a_hoc:.4f}    {a_kt:.4f}       {len(mh.support_)}")
print()

luoi = {"C": [0.1, 1, 10, 100], "gamma": [0.01, 0.1, 1, 10]}
tim = GridSearchCV(SVC(kernel="rbf"), luoi, cv=5)
tim.fit(X_tr, y_tr)

bang = pd.DataFrame(tim.cv_results_["params"])
bang["điểm"] = tim.cv_results_["mean_test_score"]
print("Điểm kiểm định chéo, hàng là C, cột là gamma")
print(bang.pivot(index="C", columns="gamma", values="điểm").round(4).to_string())
print()
print("Cặp tốt nhất:", tim.best_params_)
print(f"Điểm kiểm định chéo: {tim.best_score_:.4f}")
print(f"Accuracy trên tập kiểm tra: {tim.score(X_te, y_te):.4f}")

### Bước 5: Nhận dạng chữ số viết tay (`g5_chu_so.py`)

In [ ]:
X_dig, y_dig = load_digits(return_X_y=True)
print("Kích thước X:", X_dig.shape)
print("Độ sáng nhỏ nhất và lớn nhất:", X_dig.min(), X_dig.max())
print("Số ảnh mỗi chữ số:", np.bincount(y_dig))
print()

X_tr, X_te, y_tr, y_te = train_test_split(X_dig, y_dig, test_size=0.25, random_state=17, stratify=y_dig)
svm_dig = SVC(kernel="rbf").fit(X_tr, y_tr)
du_doan = svm_dig.predict(X_te)

print(f"SVM kernel RBF: accuracy {accuracy_score(y_te, du_doan):.4f}")
print(f"Số vectơ hỗ trợ: {len(svm_dig.support_)} trên {len(X_tr)} ảnh")
sai = np.flatnonzero(du_doan != y_te)
print(f"Số ảnh đoán sai: {len(sai)} trên {len(y_te)}")
for i in sai:
    print(f"  ảnh {i:3d}: thật là {y_te[i]}, đoán là {du_doan[i]}")
print()

mo_hinh_cs = {
    "SVM có chuẩn hóa": make_pipeline(StandardScaler(), SVC()),
    "KNN, k = 5": KNeighborsClassifier(),
    "Logistic có chuẩn hóa": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
}
print("Các mô hình khác, accuracy trên tập kiểm tra")
for ten, mh in mo_hinh_cs.items():
    mh.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, mh.predict(X_te))
    print(f"  {ten:22s} {acc:.4f}")

---  
# PHẦN II: LỜI GIẢI BÀI TẬP (TRANG 15-16) & KẾT QUẢ ĐẦU RA

### Bài tập 1: Tính tay khoảng cách tới ranh giới

In [ ]:
X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]], cluster_std=0.9, random_state=0)
y = 1 - y
svm = SVC(kernel="linear", C=1000).fit(X, y)

w = svm.coef_[0]
b = svm.intercept_[0]
do_dai_w = np.linalg.norm(w)

print(f"Hệ số: w1 = {w[0]:.3f}, w2 = {w[1]:.3f}, b = {b:.3f}")
print(f"Độ dài ||w|| = {do_dai_w:.4f}\n")

diem_moi = np.array([[3.0, 3.0], [5.0, 4.5]])
for pt in diem_moi:
    f_val = w[0] * pt[0] + w[1] * pt[1] + b
    d_val = abs(f_val) / do_dai_w
    print(f"Điểm {pt}: f(x) = {f_val:+.3f} | Khoảng cách d = |f(x)|/||w|| = {d_val:.4f}")

print("\nKhoảng cách của 3 vectơ hỗ trợ:")
for idx in svm.support_:
    f_sv = w[0] * X[idx,0] + w[1] * X[idx,1] + b
    d_sv = abs(f_sv) / do_dai_w
    print(f"Vectơ hỗ trợ {idx} ({X[idx,0]:.2f}, {X[idx,1]:.2f}): f(x) = {f_sv:+.3f} | Khoảng cách = {d_sv:.4f}")

print(f"\nĐộ rộng lề tổng cộng (2/||w||) = {2/do_dai_w:.4f}")
print("-> Giải thích: Các vectơ hỗ trợ nằm đúng trên hai mép lề f(x) = +1 và f(x) = -1. Khoảng cách từ mỗi mép lề đến đường ranh giới f(x) = 0 là 1/||w||, bằng đúng một nửa độ rộng lề tổng cộng.")

### Bài tập 2: Thêm một điểm

In [ ]:
X, y = make_blobs(n_samples=24, centers=[[2, 2], [6, 6]], cluster_std=0.9, random_state=0)
y = 1 - y

X_1 = np.vstack([X, [-1, -1]])
y_1 = np.append(y, 1)
svm1 = SVC(kernel="linear", C=1000).fit(X_1, y_1)
print(f"Lần 1 (thêm (-1,-1)): w1={svm1.coef_[0][0]:.3f}, w2={svm1.coef_[0][1]:.3f}, b={svm1.intercept_[0]:.3f}")

X_2 = np.vstack([X, [4.5, 4]])
y_2 = np.append(y, 1)
svm2 = SVC(kernel="linear", C=1000).fit(X_2, y_2)
print(f"Lần 2 (thêm (4.5,4)): w1={svm2.coef_[0][0]:.3f}, w2={svm2.coef_[0][1]:.3f}, b={svm2.intercept_[0]:.3f}")

print("\nGiải thích:")
print("- Lần 1: Điểm (-1,-1) nằm sâu trong vùng lớp 1, không trở thành vectơ hỗ trợ nên đường ranh giới KHÔNG THAY ĐỔI.")
print("- Lần 2: Điểm (4.5,4) nằm sát ranh giới, trở thành vectơ hỗ trợ mới, làm đường ranh giới THAY ĐỔI.")

### Bài tập 3: Kernel đa thức

In [ ]:
df_vt = pd.read_csv("data/hai_vang_trang.csv")
X_vt = df_vt[["x1", "x2"]]
y_vt = df_vt["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(X_vt, y_vt, test_size=0.25, random_state=17, stratify=y_vt)

for d in [2, 3, 4]:
    pipe = make_pipeline(StandardScaler(), SVC(kernel="poly", degree=d))
    grid = GridSearchCV(pipe, {"svc__C": [0.1, 1, 10, 100]}, cv=5)
    grid.fit(X_tr, y_tr)
    print(f"Bậc d={d}: C tốt nhất={grid.best_params_['svc__C']}, Điểm CV={grid.best_score_:.4f}, Accuracy={grid.score(X_te, y_te):.4f}")

print("\nNhận xét: Bậc 3 cho kết quả tốt nhất trong các bậc đa thức, nhưng KHÔNG vượt được Kernel RBF (0.9200) ở Mục 5.")

### Bài tập 4: Dò lưới mịn

In [ ]:
luoi_min = {"C": [3, 10, 30], "gamma": [0.3, 1, 3]}
tim_min = GridSearchCV(SVC(kernel="rbf"), luoi_min, cv=5)
tim_min.fit(X_tr, y_tr)

print("Cặp tốt nhất sau khi dò mịn:", tim_min.best_params_)
print(f"Điểm kiểm định chéo: {tim_min.best_score_:.4f}")
print(f"Accuracy trên tập kiểm tra: {tim_min.score(X_te, y_te):.4f}")

print("\nNhận xét: Dò mịn giúp tối ưu hóa siêu tham số cục bộ mà không tốn quá nhiều chi phí tính toán.")

### Bài tập 5: Chữ số viết tay, những lỗi hay gặp

In [ ]:
X_dig, y_dig = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X_dig, y_dig, test_size=0.25, random_state=17, stratify=y_dig)
svm_dig = SVC(kernel="rbf").fit(X_tr, y_tr)
du_doan = svm_dig.predict(X_te)

cm = confusion_matrix(y_te, du_doan)
print("Bảng nhầm lẫn (Confusion Matrix 10x10):")
print(pd.DataFrame(cm, index=[f"Thật {i}" for i in range(10)], columns=[f"Đoán {i}" for i in range(10)]))
print()

sai_idx = np.flatnonzero(du_doan != y_te)
print(f"Số ảnh đoán sai: {len(sai_idx)}/{len(y_te)}")
for idx in sai_idx:
    print(f"Ảnh chỉ số {idx:3d}: Thật là {y_te[idx]}, Đoán là {du_doan[idx]}")
    plt.figure(figsize=(1.5, 1.5))
    plt.imshow(X_te[idx].reshape(8, 8), cmap="gray_r")
    plt.title(f"Thật: {y_te[idx]} | Đoán: {du_doan[idx]}")
    plt.axis("off")
    plt.show()

print("Nhận xét: Các ảnh đoán sai đều có đặc trưng nét chữ mờ, nét viết biến dạng hoặc nét bị đứt.")

### Bài tập 6: SVM trên dữ liệu lớn

In [ ]:
sizes = [2000, 5000, 10000, 20000]
ket_qua = []

print("Đang thực thi đo thời gian...")
for n in sizes:
    X_cl, y_cl = make_classification(n_samples=n, n_features=20, random_state=0)
    
    t0 = time.time()
    SVC(kernel="rbf").fit(X_cl, y_cl)
    t_svc = time.time() - t0
    
    t0 = time.time()
    LinearSVC(max_iter=5000, random_state=0).fit(X_cl, y_cl)
    t_linearsvc = time.time() - t0
    
    ket_qua.append({"Số mẫu (n)": n, "SVC (s)": round(t_svc, 4), "LinearSVC (s)": round(t_linearsvc, 4)})

df_res = pd.DataFrame(ket_qua)
print("\nBảng so sánh thời gian huấn luyện:")
print(df_res.to_string(index=False))

print("\nNhận xét:")
print("- SVC() (kernel RBF): Thời gian học tăng khoảng 3 - 4 lần khi dữ liệu tăng gấp đôi (độ phức tạp O(n^2) đến O(n^3)).")
print("- LinearSVC(): Thời gian học tăng tuyến tính khoảng 2 lần khi dữ liệu tăng gấp đôi (độ phức tạp O(n)).")